In [1]:
import numpy as np
import torch
from utils.motion import recover_from_ric, recover_from_rot
from utils.skel import Skeleton
import matplotlib
matplotlib.use("Agg")
from mld.data.humanml.utils.plot_script import plot_3d_motion
import os

In [26]:
aeroplane = np.load("/source/junhyuk/motion-style/style-salad/dataset/100style/new_joint_vecs/030003.npy")
aeroplane_t = torch.from_numpy(aeroplane).float().to('cuda')

In [27]:
zombie = np.load("/source/junhyuk/motion-style/style-salad/dataset/100style/new_joint_vecs/037971.npy")
zombie_t = torch.from_numpy(zombie).float().to('cuda')

In [4]:
mean = torch.tensor(np.load('./checkpoints/t2m/Comp_v6_KLD005/meta/mean.npy'), dtype=torch.float32, device='cuda')
std  = torch.tensor(np.load('./checkpoints/t2m/Comp_v6_KLD005/meta/std.npy'),  dtype=torch.float32, device='cuda')

In [5]:
kinematic_tree = [
    [0, 2, 5, 8, 11],
    [0, 1, 4, 7, 10],
    [0, 3, 6, 9, 12, 15],
    [9, 14, 17, 19, 21],
    [9, 13, 16, 18, 20],
]

In [7]:
save_path = os.path.join("./zelete/aeroplane.mp4")

In [8]:
plot_3d_motion(
    save_path,
    kinematic_tree,
    joints,
    title="",
    dataset="humanml",
    fps=20,
)

In [10]:
t2m_raw_offsets = np.array([
    [0,0,0],
    [1,0,0],  [-1,0,0], [0,1,0],
    [0,-1,0], [0,-1,0], [0, 1,0],
    [0,-1,0], [0,-1,0], [0, 1,0],
    [0,0,1],  [0,0,1],  [0, 1,0],
    [1,0,0],  [-1,0,0], [0,0,1],
    [0,-1,0], [0,-1,0], [0,-1,0],
    [0,-1,0], [0,-1,0], [0,-1,0],
], dtype=np.float32)

In [11]:
t2m_kinematic_chain = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15], [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]

In [29]:
def chain_to_parents(chains, joints_num):
    parents = [-1] * joints_num
    for chain in chains:
        for p, c in zip(chain[:-1], chain[1:]):
            parents[c] = p
    return parents

parents = chain_to_parents(t2m_kinematic_chain, 22)
lengths = np.linalg.norm(t2m_raw_offsets, axis=-1).astype(np.float32)
skeleton = Skeleton(parents, lengths)

In [62]:
mean = torch.tensor(np.load('./checkpoints/t2m/Comp_v6_KLD005/meta/mean.npy'), dtype=torch.float32, device='cuda')
std  = torch.tensor(np.load('./checkpoints/t2m/Comp_v6_KLD005/meta/std.npy'),  dtype=torch.float32, device='cuda')

In [70]:
motion = aeroplane_t * std + mean
# joints = recover_from_ric(motion, 22).detach().cpu().numpy()
joints = recover_from_rot(motion, 22, skeleton).detach().cpu().numpy()

In [71]:
save_path = os.path.join("./zelete/aeroplane_rot_norm.mp4")

In [72]:
plot_3d_motion(
    save_path,
    kinematic_tree,
    joints,
    title="",
    dataset="humanml",
    fps=20,
)

In [43]:
print(joints.mean(), joints.std())

tensor(0.1982, device='cuda:0') tensor(0.6843, device='cuda:0')
